# Práctica 09 — Tests de hipótesis III: categorías y comparaciones múltiples

**Curso:** Estadística Aplicada con Python · **Clase 09** · **Tiempo estimado:** 60-90 minutos
**Notebook de clase:** `notebooks_clase/clase09_tests_hipotesis_iii.ipynb`

## Cómo funciona esta práctica

- Cada ejercicio tiene una celda con `# TODO`: complétala.
- Al final de cada ejercicio hay una **comprobación automática** (`assert`). Si falla, el mensaje te indica qué revisar.
- Los **datos son sintéticos** (*ejemplos ilustrativos*) y las semillas están fijadas, así que los resultados son reproducibles.
- Las **soluciones** están al final del notebook, en una sección separada. **No las abras antes de intentarlo.**
- Ejecuta las celdas en orden.

---
## Celda 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportions_ztest

rng = np.random.default_rng(42)          # semilla fija: todos obtenemos los mismos resultados
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)
print("Librerías cargadas correctamente.")

## Ejercicio 1 — Chi-cuadrado a mano y V de Cramér

Se comparan las devoluciones de dos productos (*ejemplo ilustrativo*): de 100 unidades vendidas del producto 1, 18 se devuelven; del producto 2, 32. Con la tabla `tabla` (filas: producto; columnas: devuelto, no devuelto) calcula:

- `esperada`: la tabla de frecuencias esperadas bajo H0 (totales de fila × totales de columna / total),
- `chi2_est`, `gl` y `p_chi` (con `stats.chi2.sf`),
- `v_cramer`,
- `condicion_ok`: `True` si **todas** las frecuencias esperadas son ≥ 5.

In [ ]:
tabla = np.array([[18, 82],
                  [32, 68]])

esperada = None      # TODO
chi2_est = None      # TODO
gl = None            # TODO
p_chi = None         # TODO
v_cramer = None      # TODO
condicion_ok = None  # TODO

# Comprobación automática
assert all(v is not None for v in (esperada, chi2_est, gl, p_chi, v_cramer, condicion_ok)), "Completa las líneas marcadas con TODO"
assert np.allclose(esperada, [[25, 75], [25, 75]]), "esperada = totales de fila × totales de columna / total general"
assert np.isclose(chi2_est, 5.2267, atol=1e-3), "χ² = suma de (observada − esperada)² / esperada"
assert gl == 1, "gl = (filas − 1) · (columnas − 1)"
assert np.isclose(p_chi, 0.0222, atol=1e-3), "p = stats.chi2.sf(chi2_est, gl)"
assert np.isclose(v_cramer, 0.1617, atol=1e-3), "V = raíz de χ² / (N · (min(filas, columnas) − 1))"
assert condicion_ok is True or condicion_ok == True, "compara la mínima esperada con 5"
print(f"χ² = {chi2_est:.3f}   gl = {gl}   p = {p_chi:.4f}   V = {v_cramer:.3f}")
print("Ejercicio 1 correcto ✔")

## Ejercicio 2 — Comparaciones múltiples

Un equipo ha realizado 10 tests sobre subgrupos y ha obtenido estos p-values (*ejemplo ilustrativo*). Calcula:

- `prob_alguno`: la probabilidad de al menos un falso positivo si los 10 tests fueran independientes y todas las H0 ciertas, con α = 0,05,
- `p_bonf`: los p-values ajustados de Bonferroni (`min(1, m · p)`), como array,
- `n_sin_corregir`, `n_bonf` y `n_bh`: número de tests que rechazan H0 sin corregir, con Bonferroni y con Benjamini-Hochberg (`multipletests(..., method="fdr_bh")`), todos con α = 0,05.

In [ ]:
pvals = np.array([0.0004, 0.003, 0.009, 0.013, 0.021, 0.038, 0.046, 0.11, 0.35, 0.72])
m = len(pvals)

prob_alguno = None     # TODO
p_bonf = None          # TODO
n_sin_corregir = None  # TODO
n_bonf = None          # TODO
n_bh = None            # TODO

# Comprobación automática
assert all(v is not None for v in (prob_alguno, p_bonf, n_sin_corregir, n_bonf, n_bh)), "Completa las líneas marcadas con TODO"
assert np.isclose(prob_alguno, 1 - 0.95 ** 10), "prob = 1 − (1 − α)^m"
assert np.allclose(p_bonf, np.minimum(1, pvals * 10)), "p ajustado = min(1, m · p)"
assert n_sin_corregir == 7, "cuenta los p < 0,05"
assert n_bonf == 2, "con Bonferroni rechazas si p ≤ 0,05 / 10"
assert n_bh == 5, "usa multipletests(pvals, alpha=0.05, method='fdr_bh')[0].sum()"
print(f"P(≥ 1 falso positivo) = {prob_alguno:.3f}   rechazos: sin corregir {n_sin_corregir}, Bonferroni {n_bonf}, BH {n_bh}")
print("Ejercicio 2 correcto ✔")

## Ejercicio 3 — La condición de las esperadas y el test de Fisher

Un piloto con 25 casos compara si un cambio de proceso reduce los errores (*ejemplo ilustrativo*): grupo nuevo, 2 errores y 10 correctos; grupo antiguo, 8 errores y 5 correctos. Con la tabla `tabla3` (filas: grupo; columnas: error, correcto) calcula:

- `esp_min`: la **menor** frecuencia esperada bajo H0,
- `condicion_ok`: `True` si todas las esperadas son ≥ 5,
- `p_chi`: el p-value del chi-cuadrado **sin** corrección (`correction=False`),
- `p_yates`: el del chi-cuadrado con corrección de Yates (la opción por defecto),
- `p_fisher`: el del test exacto de Fisher.

In [ ]:
tabla3 = np.array([[2, 10],
                   [8, 5]])

esp_min = None        # TODO
condicion_ok = None   # TODO
p_chi = None          # TODO
p_yates = None        # TODO
p_fisher = None       # TODO

# Comprobación automática
assert all(v is not None for v in (esp_min, condicion_ok, p_chi, p_yates, p_fisher)), "Completa las líneas marcadas con TODO"
assert np.isclose(esp_min, 4.8), "esperada = total de fila × total de columna / total general; toma la mínima"
assert condicion_ok is False or condicion_ok == False, "la mínima esperada es < 5, así que la condición no se cumple"
assert np.isclose(p_chi, 0.0221, atol=1e-3), "stats.chi2_contingency(tabla3, correction=False).pvalue"
assert np.isclose(p_yates, 0.0602, atol=1e-3), "stats.chi2_contingency(tabla3).pvalue"
assert np.isclose(p_fisher, 0.0414, atol=1e-3), "stats.fisher_exact(tabla3).pvalue"
print(f"esperada mínima = {esp_min:.2f}   χ² p = {p_chi:.4f}   Yates p = {p_yates:.4f}   Fisher p = {p_fisher:.4f}")
print("Ejercicio 3 correcto ✔ (¡las tres conclusiones no coinciden a α = 0,05!)")

## Ejercicio 4 — Benjamini-Hochberg a mano

Implementa Benjamini-Hochberg **sin** usar `multipletests` y compáralo con `multipletests`. Completa la función `bh_a_mano(pvalores, alfa)`, que devuelve la tupla `(rechaza, p_ajustado)`:

1. ordena los p-values (`np.argsort`);
2. el p ajustado del i-ésimo ordenado es `p_(i) · m / i`, pero tomando el **mínimo acumulado desde el final** (para que sea monótono) y con tope 1;
3. rechaza donde el p ajustado sea ≤ `alfa` (equivale a la regla "el mayor i con p_(i) ≤ (i/m)·α");
4. devuelve los resultados **en el orden original** de `pvalores`.

Después calcula `n_bh`: cuántas hipótesis rechaza BH con α = 0,05 sobre `pv12` (12 p-values, no ordenados).

In [ ]:
def bh_a_mano(pvalores, alfa=0.05):
    p = np.asarray(pvalores, dtype=float)
    # TODO: ordena, calcula p_(i)·m/i, mínimo acumulado desde el final, tope 1, y devuelve (rechaza, p_ajustado) en el orden original
    return None

pv12 = np.array([0.0433, 0.0017, 0.4278, 0.016, 0.5457, 0.0047, 0.2102, 0.0373, 0.7505, 0.0238, 0.1669, 0.5103])
res_bh = bh_a_mano(pv12)
n_bh = None      # TODO

# Comprobación automática
assert res_bh is not None and n_bh is not None, "Completa las líneas marcadas con TODO"
rechaza, p_aj = res_bh
r_ref = multipletests(pv12, alpha=0.05, method="fdr_bh")
assert np.allclose(p_aj, r_ref[1]), "el p ajustado debe coincidir con multipletests(..., method='fdr_bh')[1]"
assert np.array_equal(rechaza, r_ref[0]), "rechaza = p_ajustado <= alfa"
otro = np.random.default_rng(1).uniform(size=30) ** 2
assert np.allclose(bh_a_mano(otro)[1], multipletests(otro, method="fdr_bh")[1]), "tu función debe funcionar con cualquier vector de p-values"
assert n_bh == 2, "n_bh es el número de True en rechaza"
print(f"BH rechaza {n_bh} de {len(pv12)};  Bonferroni: {(pv12 <= 0.05 / len(pv12)).sum()};  sin corregir: {(pv12 < 0.05).sum()}")
print("Ejercicio 4 correcto ✔")

## Ejercicio 5 — z² = χ², con verificación numérica

Una página con el diseño A convierte 52 de 400 visitas y con el diseño B, 78 de 400 (*ejemplo ilustrativo*). Calcula:

- `z` y `p_z`: el estadístico y el p-value de `proportions_ztest` (dos colas), con `conteos = [52, 78]` y `tamanos = [400, 400]`,
- `chi2_val` y `p_chi`: el estadístico y el p-value de `chi2_contingency` **sin** corrección, sobre la tabla 2×2 (éxitos y fracasos por diseño),
- `chi2_yates`: el estadístico **con** corrección de Yates,
- `coinciden`: `True` si z² y χ² (sin corrección) son iguales dentro de un error numérico (`np.isclose`).

In [ ]:
conteos = [52, 78]
tamanos = [400, 400]
tabla5 = np.array([[52, 348],
                   [78, 322]])

z = None            # TODO
p_z = None          # TODO
chi2_val = None     # TODO
p_chi = None        # TODO
chi2_yates = None   # TODO
coinciden = None    # TODO

# Comprobación automática
assert all(v is not None for v in (z, p_z, chi2_val, p_chi, chi2_yates, coinciden)), "Completa las líneas marcadas con TODO"
assert np.isclose(z, -2.4918, atol=1e-3) and np.isclose(p_z, 0.0127, atol=1e-3), "usa proportions_ztest(conteos, tamanos)"
assert np.isclose(chi2_val, 6.2090, atol=1e-3) and np.isclose(p_chi, 0.0127, atol=1e-3), "usa stats.chi2_contingency(tabla5, correction=False)"
assert coinciden is True or coinciden == True, "compara z ** 2 con chi2_val"
assert chi2_yates < chi2_val, "la corrección de Yates es más prudente: usa stats.chi2_contingency(tabla5)"
print(f"z = {z:.4f}   z² = {z ** 2:.4f}   χ² = {chi2_val:.4f}   p(z) = {p_z:.4f}   p(χ²) = {p_chi:.4f}   χ² Yates = {chi2_yates:.4f}")
print("Ejercicio 5 correcto ✔")

---
## Reto final — Un informe de tests con corrección de comparaciones múltiples

Un equipo compara la conversión de dos canales en 10 segmentos de clientes (*ejemplo ilustrativo*; los primeros 6 segmentos tienen 150 visitas por canal, los 4 últimos son pequeños). Completa la función `informe_tests(tablas)`, que recibe una lista de tablas 2×2 y devuelve un diccionario con las claves:

- `"p"`: array con el p-value de cada tabla, usando el **chi-cuadrado sin corrección** si todas las esperadas son ≥ 5 y el **test exacto de Fisher** si no,
- `"n_fisher"`: cuántas tablas han necesitado Fisher,
- `"n_sin_corregir"`, `"n_bonf"`, `"n_bh"`: número de tests que rechazan con α = 0,05 sin corregir, con Bonferroni y con Benjamini-Hochberg (sobre **toda** la familia de tests).

Después elige en `respuesta` la única interpretación correcta:

- `"a"`: los tests con p < 0,05 sin corregir son todos hallazgos reales.
- `"b"`: al corregir por la familia completa de 10 tests se mantienen menos resultados que sin corregir; el hallazgo que desaparece es compatible con un falso positivo, y la corrección tiene un coste en potencia.
- `"c"`: un p-value de 0,004 significa que la probabilidad de que la diferencia no sea real es del 0,4 %.

In [ ]:
rng_r = np.random.default_rng(910)
tablas = []
for k, n in enumerate([150, 150, 150, 150, 150, 150, 20, 20, 15, 15]):
    a = rng_r.binomial(n, 0.20)
    b = rng_r.binomial(n, 0.34 if k in (0, 1) else 0.20)
    tablas.append(np.array([[a, n - a], [b, n - b]]))

def informe_tests(tablas):
    """Devuelve p, n_fisher, n_sin_corregir, n_bonf y n_bh."""
    # TODO: para cada tabla, χ² sin corrección o Fisher según las esperadas; luego cuenta los rechazos
    return None

inf = informe_tests(tablas)
respuesta = None    # TODO: "a", "b" o "c"

# Comprobación automática
assert inf is not None and respuesta is not None, "Completa la función y la respuesta (TODO)"
assert set(inf) == {"p", "n_fisher", "n_sin_corregir", "n_bonf", "n_bh"}, "el diccionario debe tener exactamente las claves p, n_fisher, n_sin_corregir, n_bonf y n_bh"
assert len(inf["p"]) == 10 and np.isclose(inf["p"][0], 0.0004, atol=1e-4) and np.isclose(inf["p"][7], 0.0471, atol=1e-3), "usa chi2_contingency(t, correction=False) si la mínima esperada es ≥ 5 y fisher_exact(t) si no"
assert inf["n_fisher"] == 3, "cuenta las tablas con mínima esperada < 5"
assert inf["n_sin_corregir"] == 3, "cuenta los p < 0,05"
assert inf["n_bonf"] == 2 and inf["n_bh"] == 2, "aplica multipletests a los 10 p-values (bonferroni y fdr_bh)"
assert respuesta == "b", "el p-value no es la probabilidad de que H0 sea cierta, y no corregir infla los falsos positivos"
print(f"tablas con Fisher: {inf['n_fisher']}   rechazos: sin corregir {inf['n_sin_corregir']}, Bonferroni {inf['n_bonf']}, BH {inf['n_bh']}")
print("Reto final correcto ✔")

---
## Conclusión de la práctica

Has calculado un chi-cuadrado y la V de Cramér a mano, has comprobado la condición de las esperadas y usado Fisher, has implementado Benjamini-Hochberg, has visto que z² = χ² y has escrito un informe de tests con corrección de comparaciones múltiples. Siguiente paso: la clase 10 (correlación y regresión lineal).

---
## Soluciones (¡no mires antes de haberlo intentado!)

Cada celda siguiente contiene la solución de un ejercicio. Puedes ejecutarlas para comparar con tu resultado.

### Solución del ejercicio 1

In [ ]:
tabla = np.array([[18, 82],
                  [32, 68]])

esperada = tabla.sum(axis=1, keepdims=True) * tabla.sum(axis=0, keepdims=True) / tabla.sum()
chi2_est = ((tabla - esperada) ** 2 / esperada).sum()
gl = (tabla.shape[0] - 1) * (tabla.shape[1] - 1)
p_chi = stats.chi2.sf(chi2_est, gl)
v_cramer = np.sqrt(chi2_est / (tabla.sum() * (min(tabla.shape) - 1)))
condicion_ok = bool(esperada.min() >= 5)

assert all(v is not None for v in (esperada, chi2_est, gl, p_chi, v_cramer, condicion_ok)), "Completa las líneas marcadas con TODO"
assert np.allclose(esperada, [[25, 75], [25, 75]]), "esperada = totales de fila × totales de columna / total general"
assert np.isclose(chi2_est, 5.2267, atol=1e-3), "χ² = suma de (observada − esperada)² / esperada"
assert gl == 1, "gl = (filas − 1) · (columnas − 1)"
assert np.isclose(p_chi, 0.0222, atol=1e-3), "p = stats.chi2.sf(chi2_est, gl)"
assert np.isclose(v_cramer, 0.1617, atol=1e-3), "V = raíz de χ² / (N · (min(filas, columnas) − 1))"
assert condicion_ok is True or condicion_ok == True, "compara la mínima esperada con 5"
print(f"χ² = {chi2_est:.3f}   gl = {gl}   p = {p_chi:.4f}   V = {v_cramer:.3f}")
print("Ejercicio 1 correcto ✔")

### Solución del ejercicio 2

In [ ]:
pvals = np.array([0.0004, 0.003, 0.009, 0.013, 0.021, 0.038, 0.046, 0.11, 0.35, 0.72])
m = len(pvals)

prob_alguno = 1 - (1 - 0.05) ** m
p_bonf = np.minimum(1, m * pvals)
n_sin_corregir = int((pvals < 0.05).sum())
n_bonf = int((p_bonf < 0.05).sum())
n_bh = int(multipletests(pvals, alpha=0.05, method="fdr_bh")[0].sum())

assert all(v is not None for v in (prob_alguno, p_bonf, n_sin_corregir, n_bonf, n_bh)), "Completa las líneas marcadas con TODO"
assert np.isclose(prob_alguno, 1 - 0.95 ** 10), "prob = 1 − (1 − α)^m"
assert np.allclose(p_bonf, np.minimum(1, pvals * 10)), "p ajustado = min(1, m · p)"
assert n_sin_corregir == 7, "cuenta los p < 0,05"
assert n_bonf == 2, "con Bonferroni rechazas si p ≤ 0,05 / 10"
assert n_bh == 5, "usa multipletests(pvals, alpha=0.05, method='fdr_bh')[0].sum()"
print(f"P(≥ 1 falso positivo) = {prob_alguno:.3f}   rechazos: sin corregir {n_sin_corregir}, Bonferroni {n_bonf}, BH {n_bh}")
print("Ejercicio 2 correcto ✔")

### Solución del ejercicio 3

In [ ]:
tabla3 = np.array([[2, 10],
                   [8, 5]])

esp_min = stats.chi2_contingency(tabla3).expected_freq.min()
condicion_ok = bool(esp_min >= 5)
p_chi = stats.chi2_contingency(tabla3, correction=False).pvalue
p_yates = stats.chi2_contingency(tabla3).pvalue
p_fisher = stats.fisher_exact(tabla3).pvalue

assert all(v is not None for v in (esp_min, condicion_ok, p_chi, p_yates, p_fisher)), "Completa las líneas marcadas con TODO"
assert np.isclose(esp_min, 4.8), "esperada = total de fila × total de columna / total general; toma la mínima"
assert condicion_ok is False or condicion_ok == False, "la mínima esperada es < 5, así que la condición no se cumple"
assert np.isclose(p_chi, 0.0221, atol=1e-3), "stats.chi2_contingency(tabla3, correction=False).pvalue"
assert np.isclose(p_yates, 0.0602, atol=1e-3), "stats.chi2_contingency(tabla3).pvalue"
assert np.isclose(p_fisher, 0.0414, atol=1e-3), "stats.fisher_exact(tabla3).pvalue"
print(f"esperada mínima = {esp_min:.2f}   χ² p = {p_chi:.4f}   Yates p = {p_yates:.4f}   Fisher p = {p_fisher:.4f}")
print("Ejercicio 3 correcto ✔ (¡las tres conclusiones no coinciden a α = 0,05!)")

### Solución del ejercicio 4

In [ ]:
def bh_a_mano(pvalores, alfa=0.05):
    p = np.asarray(pvalores, dtype=float)
    m = len(p)
    orden = np.argsort(p)
    aj_orden = np.minimum.accumulate((p[orden] * m / np.arange(1, m + 1))[::-1])[::-1]
    aj_orden = np.minimum(aj_orden, 1.0)
    p_ajustado = np.empty(m)
    p_ajustado[orden] = aj_orden
    return p_ajustado <= alfa, p_ajustado

pv12 = np.array([0.0433, 0.0017, 0.4278, 0.016, 0.5457, 0.0047, 0.2102, 0.0373, 0.7505, 0.0238, 0.1669, 0.5103])
res_bh = bh_a_mano(pv12)
n_bh = int(res_bh[0].sum())

assert res_bh is not None and n_bh is not None, "Completa las líneas marcadas con TODO"
rechaza, p_aj = res_bh
r_ref = multipletests(pv12, alpha=0.05, method="fdr_bh")
assert np.allclose(p_aj, r_ref[1]), "el p ajustado debe coincidir con multipletests(..., method='fdr_bh')[1]"
assert np.array_equal(rechaza, r_ref[0]), "rechaza = p_ajustado <= alfa"
otro = np.random.default_rng(1).uniform(size=30) ** 2
assert np.allclose(bh_a_mano(otro)[1], multipletests(otro, method="fdr_bh")[1]), "tu función debe funcionar con cualquier vector de p-values"
assert n_bh == 2, "n_bh es el número de True en rechaza"
print(f"BH rechaza {n_bh} de {len(pv12)};  Bonferroni: {(pv12 <= 0.05 / len(pv12)).sum()};  sin corregir: {(pv12 < 0.05).sum()}")
print("Ejercicio 4 correcto ✔")

### Solución del ejercicio 5

In [ ]:
conteos = [52, 78]
tamanos = [400, 400]
tabla5 = np.array([[52, 348],
                   [78, 322]])

z, p_z = proportions_ztest(conteos, tamanos)
res5 = stats.chi2_contingency(tabla5, correction=False)
chi2_val, p_chi = res5.statistic, res5.pvalue
chi2_yates = stats.chi2_contingency(tabla5).statistic
coinciden = bool(np.isclose(z ** 2, chi2_val))

assert all(v is not None for v in (z, p_z, chi2_val, p_chi, chi2_yates, coinciden)), "Completa las líneas marcadas con TODO"
assert np.isclose(z, -2.4918, atol=1e-3) and np.isclose(p_z, 0.0127, atol=1e-3), "usa proportions_ztest(conteos, tamanos)"
assert np.isclose(chi2_val, 6.2090, atol=1e-3) and np.isclose(p_chi, 0.0127, atol=1e-3), "usa stats.chi2_contingency(tabla5, correction=False)"
assert coinciden is True or coinciden == True, "compara z ** 2 con chi2_val"
assert chi2_yates < chi2_val, "la corrección de Yates es más prudente: usa stats.chi2_contingency(tabla5)"
print(f"z = {z:.4f}   z² = {z ** 2:.4f}   χ² = {chi2_val:.4f}   p(z) = {p_z:.4f}   p(χ²) = {p_chi:.4f}   χ² Yates = {chi2_yates:.4f}")
print("Ejercicio 5 correcto ✔")

### Solución del reto final

In [ ]:
rng_r = np.random.default_rng(910)
tablas = []
for k, n in enumerate([150, 150, 150, 150, 150, 150, 20, 20, 15, 15]):
    a = rng_r.binomial(n, 0.20)
    b = rng_r.binomial(n, 0.34 if k in (0, 1) else 0.20)
    tablas.append(np.array([[a, n - a], [b, n - b]]))

def informe_tests(tablas):
    """Devuelve p, n_fisher, n_sin_corregir, n_bonf y n_bh."""
    p, n_fisher = [], 0
    for t in tablas:
        c = stats.chi2_contingency(t, correction=False)
        if c.expected_freq.min() < 5:
            p.append(stats.fisher_exact(t).pvalue)
            n_fisher += 1
        else:
            p.append(c.pvalue)
    p = np.array(p)
    return {"p": p, "n_fisher": n_fisher, "n_sin_corregir": int((p < 0.05).sum()),
            "n_bonf": int(multipletests(p, method="bonferroni")[0].sum()),
            "n_bh": int(multipletests(p, method="fdr_bh")[0].sum())}

inf = informe_tests(tablas)
respuesta = "b"

assert inf is not None and respuesta is not None, "Completa la función y la respuesta (TODO)"
assert set(inf) == {"p", "n_fisher", "n_sin_corregir", "n_bonf", "n_bh"}, "el diccionario debe tener exactamente las claves p, n_fisher, n_sin_corregir, n_bonf y n_bh"
assert len(inf["p"]) == 10 and np.isclose(inf["p"][0], 0.0004, atol=1e-4) and np.isclose(inf["p"][7], 0.0471, atol=1e-3), "usa chi2_contingency(t, correction=False) si la mínima esperada es ≥ 5 y fisher_exact(t) si no"
assert inf["n_fisher"] == 3, "cuenta las tablas con mínima esperada < 5"
assert inf["n_sin_corregir"] == 3, "cuenta los p < 0,05"
assert inf["n_bonf"] == 2 and inf["n_bh"] == 2, "aplica multipletests a los 10 p-values (bonferroni y fdr_bh)"
assert respuesta == "b", "el p-value no es la probabilidad de que H0 sea cierta, y no corregir infla los falsos positivos"
print(f"tablas con Fisher: {inf['n_fisher']}   rechazos: sin corregir {inf['n_sin_corregir']}, Bonferroni {inf['n_bonf']}, BH {inf['n_bh']}")
print("Reto final correcto ✔")

---
## Autoevaluación de la clase

Responde con honestidad (una o dos frases cada una):

1. ¿Qué ejercicio te ha costado más y por qué?
2. ¿Qué concepto de la clase no te ha quedado claro (esperadas del chi-cuadrado, z² = χ², cuándo Fisher, FWER frente a FDR, cómo corregir un post-hoc)?
3. ¿Qué cambiarías o repetirías la próxima vez?
4. ¿Cuánto tiempo has dedicado en total?

*(Tus respuestas aquí.)*